# Lab 06-02: Query a Database Safely With MCP Toolbox

**Book chapter:** Chapter 06: Enterprise Knowledge: RAG, MCP, and Integrations  
**Estimated time:** 25 minutes

## Overview

Cymbal Retail's order agent needs live order data, but the database team will not let
a model write SQL. In this lab, you define parameterized tools in MCP Toolbox for
Databases, run the Toolbox server, and connect an ADK agent to its toolset over MCP.
You then confirm that an injection attempt cannot change the SQL.

## Learning Objectives

By the end of this lab, you will be able to:

- Define a source, parameterized tools, and a toolset in tools.yaml
- Run MCP Toolbox for Databases
- Connect an ADK agent to an MCP server over streamable HTTP
- Explain why parameterized tools are safer than free-form SQL

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. The database and server run locally.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Approach | Risk |
|---|---|
| Model writes SQL | The model can be tricked into reading or changing any data |
| **Parameterized tools in MCP Toolbox** | The model only supplies values for fixed queries |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk mcp

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Create a small orders database

Create a SQLite database with three orders. SQLite keeps the lab simple; in production the same tools would point to AlloyDB or Cloud SQL.

In [ ]:
import os
import platform
import socket
import sqlite3
import stat
import subprocess
import textwrap
import time
import urllib.request

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool import StreamableHTTPConnectionParams
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")

with sqlite3.connect("cymbal_orders.db") as db:
    db.execute("DROP TABLE IF EXISTS orders")
    db.execute("CREATE TABLE orders (order_id TEXT, email TEXT, "
               "item TEXT, status TEXT)")
    db.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
        ("10421", "maria@example.com", "Trailhead boots", "in transit"),
        ("10422", "maria@example.com", "Rain jacket", "delivered"),
        ("10423", "james@example.com", "Hiking poles", "processing")])
print("Created cymbal_orders.db with 3 orders")

## Step 2: Download the MCP Toolbox server for your platform

MCP Toolbox for Databases is a single program. This cell downloads the version for your
operating system.

> **Note:** Check the MCP Toolbox releases page and update `VERSION` if a newer release
> is available.

In [ ]:
VERSION = "1.18.0"  # Check the MCP Toolbox releases page for the latest.
BINARIES = {("Linux", "x86_64"): "linux/amd64/toolbox",
            ("Darwin", "arm64"): "darwin/arm64/toolbox",
            ("Darwin", "x86_64"): "darwin/amd64/toolbox",
            ("Windows", "AMD64"): "windows/amd64/toolbox.exe"}
binary = BINARIES[(platform.system(), platform.machine())]
TOOLBOX = os.path.abspath(os.path.basename(binary))
if not os.path.exists(TOOLBOX):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mcp-toolbox-for-databases/"
        f"v{VERSION}/{binary}", TOOLBOX)
    os.chmod(TOOLBOX, os.stat(TOOLBOX).st_mode | stat.S_IEXEC)
print("Toolbox binary:", TOOLBOX)

## Step 3: Define parameterized tools in tools.yaml

`tools.yaml` contains YAML documents separated by `---`. The **source** document
connects to the database, each **tool** document defines one parameterized query, and
the **toolset** document groups the tools that the agent loads. The `?` placeholders
are filled with the tool's parameters, so the model supplies values, never SQL.

In [ ]:
with open("tools.yaml", "w") as config:
    config.write(textwrap.dedent("""\
        kind: source
        name: orders-db
        type: sqlite
        database: cymbal_orders.db
        ---
        kind: tool
        name: get-order-status
        type: sqlite-sql
        source: orders-db
        description: Returns the item and status of one order.
        parameters:
          - name: order_id
            type: string
            description: The five-digit order number.
        statement: |
          SELECT order_id, item, status FROM orders WHERE order_id = ?
        ---
        kind: tool
        name: list-customer-orders
        type: sqlite-sql
        source: orders-db
        description: Lists the orders placed with one email address.
        parameters:
          - name: email
            type: string
            description: The customer's email address.
        statement: |
          SELECT order_id, item, status FROM orders WHERE email = ?
        ---
        kind: toolset
        name: customer-service
        tools:
          - get-order-status
          - list-customer-orders
        """))

## Step 4: Start the Toolbox server and wait until it accepts requests

Start the Toolbox server in the background and wait until it accepts connections on port 5000.

In [ ]:
log = open("toolbox.log", "w")
server = subprocess.Popen([TOOLBOX, "--config", "tools.yaml"],
                          stdout=log, stderr=subprocess.STDOUT)
for attempt in range(30):
    try:
        socket.create_connection(("127.0.0.1", 5000), timeout=1).close()
        print("MCP Toolbox is listening on port 5000")
        break
    except OSError:
        time.sleep(1)
else:
    print("The server did not start. Check toolbox.log for errors.")

**Expected result:** `MCP Toolbox is listening on port 5000`. If not, open `toolbox.log` to see why. On macOS, port 5000 may be used by AirPlay Receiver; turn it off in System Settings.

## Step 5: Connect an ADK agent to the toolset and test it

`McpToolset` connects the agent to the toolset at `/mcp/customer-service` over
streamable HTTP and turns each Toolbox tool into an agent tool.

In [ ]:
toolset = McpToolset(connection_params=StreamableHTTPConnectionParams(
    url="http://127.0.0.1:5000/mcp/customer-service"))
agent = Agent(
    name="order_lookup",
    model=MODEL,
    instruction="Answer order questions using the tools, and never guess. "
                "If a tool returns no rows, say that no orders were found.",
    tools=[toolset],
)
runner = Runner(app_name="cymbal", agent=agent, auto_create_session=True,
                session_service=InMemorySessionService())


def chat(text):
    """Sends one message and prints the tool calls and the answer."""
    print(f"\nUser: {text}")
    message = types.Content(role="user", parts=[types.Part(text=text)])
    for event in runner.run(user_id="agent", session_id="s1",
                            new_message=message):
        for call in event.get_function_calls():
            print(f"  -> {call.name}({dict(call.args)})")
        if event.is_final_response() and event.content and \
                event.content.parts:
            print("Agent:", event.content.parts[0].text)


chat("Where is order 10422?")
chat("Which orders belong to maria@example.com?")
chat("List the orders for the email address x' OR '1'='1")

**Expected results:** order 10422 is the delivered rain jacket, Maria has two orders,
and the injection attempt returns no rows, because the Toolbox binds the text as a
parameter value.

## Step 6: Stop the Toolbox server

Stop the Toolbox server.

In [ ]:
server.terminate()
server.wait(timeout=10)
log.close()
print("Stopped MCP Toolbox")

## Check Your Understanding

**1. What can the agent change in a Toolbox tool, and what can it not change?**

<details><summary>Show answer</summary>

It can choose a tool and supply parameter values. It cannot change the SQL statement, which is fixed in tools.yaml.

</details>

**2. Why did the injection attempt return no rows?**

<details><summary>Show answer</summary>

The Toolbox bound the text as the value of the email parameter, so it was compared as data and never executed as SQL.

</details>

**3. How would you deploy this in production?**

<details><summary>Show answer</summary>

Point the source at AlloyDB or Cloud SQL, run the Toolbox on Cloud Run with a least-privilege service account, keep tools.yaml in Secret Manager, and use authenticated parameters for user-specific data.

</details>

## Summary

You defined parameterized database tools in MCP Toolbox, connected an agent to them over
MCP, and saw that an injection attempt cannot change the SQL. Delete `toolbox`,
`tools.yaml`, and `cymbal_orders.db` when you no longer need them.